In [ ]:
# compute_handeye_transform.py
import numpy as np
import json
import os

HAND_FILE = "calibration/output/handeye_points.json"
OUT = "calibration/output/camera_to_robot_affine.npy"

if not os.path.exists(HAND_FILE):
    print("No hand-eye points found. Run collect_handeye_points.py first.")
    exit(1)

pairs = json.load(open(HAND_FILE, "r"))
if len(pairs) < 3:
    print("Need at least 3 point pairs for a stable affine solve; collected:", len(pairs))
    exit(1)

# Build least squares system: for each pair
# [ cx cy 1  0  0  0 ] [a11]   [rx]
# [ 0  0  0  cx cy 1 ] [a21] = [ry]
A_rows = []
b = []
for p in pairs:
    cx, cy = p["cam"]
    rx, ry = p["robot"]
    A_rows.append([cx, cy, 1, 0, 0, 0])
    A_rows.append([0, 0, 0, cx, cy, 1])
    b.append(rx)
    b.append(ry)

A = np.array(A_rows)
b = np.array(b)

# solve least squares
x, resid, rank, s = np.linalg.lstsq(A, b, rcond=None)
# x is length 6: [a11, a12, tx, a21, a22, ty]
affine = np.array([[x[0], x[1], x[2]],
                   [x[3], x[4], x[5]]])
np.save(OUT, affine)
print("Saved camera->robot affine transform to", OUT)
print("Affine matrix:\n", affine)
print("Residual norm:", resid)
